# 03 — Beta-Series GLM (LSS)

**Purpose:** Fit the beta-series model and save one beta map per trial. This is the core analysis notebook.

---

## The LSS approach

Beta-series analysis uses the **Least Squares Separate (LSS)** method. The idea is simple but requires running many GLMs:

- You have N trials (e.g., 32 feel trials per run)
- For **each trial i**, you fit a separate GLM with a specially constructed design matrix:
  - Trial i gets its own regressor: this is the **trial of interest**
  - All other trials of the **same condition** as trial i are bundled into one "nuisance" regressor
  - All trials from **other conditions** keep their own condition-level regressors
  - Confounds and drift terms are included as usual
- You extract the beta estimate for the trial-of-interest regressor and save it as a brain image
- After N GLMs, you have N beta maps — one per trial

Why LSS instead of putting all unique trial regressors in one big GLM (LSA)? Because when many trials from the same condition share a design matrix, their regressors become correlated, which inflates the variance of the estimates. LSS avoids this by isolating one trial at a time.

---

## Output structure

Each model run (unique combination of parameters) saves to a timestamped folder:
```
Analysis/outputs/{subject}/beta_series/runs/{run_id}/
    beta_maps/          # one .nii.gz per trial, per run
    design_matrices/    # the design matrix used for each LSS GLM (optional, for inspection)
    figures/            # any QC plots
    provenance/         # params.json, inputs.tsv, copy of this notebook
```

---
## Section 1 — Configuration and Parameters

Set the subject and all model parameters here. Here are all the input paths you'll need for `sub-097`:

**BOLD images:**
```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
```

**Brain mask** (you only need one — use modulate1's):
```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz
```

**Confounds:**
```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_desc-confounds_timeseries.tsv
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_desc-confounds_timeseries.tsv
```

**Beta-series labeled events** (produced by notebook 01):
```
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\sub-097_task-modulate1_events-betaSeries.tsv
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\sub-097_task-modulate2_events-betaSeries.tsv
```

**Output run directory** (the `{run_id}` part is generated at runtime from a timestamp):
```
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\runs\{run_id}\
    beta_maps\
    design_matrices\
    figures\
    provenance\
```

Building the timestamped run ID and creating all output subdirectories:

```python
run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_label = f"tR-{t_r:g}s__smooth-{smoothing_fwhm:g}mm__conf-{len(confound_columns)}"
run_id    = f"{run_stamp}__{run_label}"

run_dir           = beta_series_dir / "runs" / run_id
beta_maps_dir     = run_dir / "beta_maps"
design_mats_dir   = run_dir / "design_matrices"
figures_dir       = run_dir / "figures"
provenance_dir    = run_dir / "provenance"

for folder in [beta_maps_dir, design_mats_dir, figures_dir, provenance_dir]:
    folder.mkdir(parents=True, exist_ok=True)
```

**TODO:**
- [ ] Define all input paths from the subject variables
- [ ] Build `run_id` and all output subdirectories using the pattern above

In [18]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import pandas as pd

# ============================================================
# CONFIGURATION — change only this section when re-running
# ============================================================
subject = "sub-097"
tasks = ["modulate1", "modulate2"]

t_r = 2.0
smoothing_fwhm = 6.0
hrf_model = "spm"
confound_columns = [
    "trans_x", "trans_y", "trans_z",
    "rot_x",   "rot_y",   "rot_z",
]
# ============================================================
# TODO: define project_dir, derivatives paths, beta_series_dir
project_dir = Path(r"C:\ManzaRotation")
func_deriv_dir = project_dir / "Derivatives" / subject / "func"
beta_series_dir = project_dir / "Analysis" / "outputs" / subject / "beta_series"

bold_paths = {
    task: func_deriv_dir / f"{subject}_task-{task}_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz"
    for task in tasks
}
bold1_path = bold_paths["modulate1"]
bold2_path = bold_paths["modulate2"]

mask_path = func_deriv_dir / f"{subject}_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz"

beta_events1_path = beta_series_dir / "tables" / f"{subject}_modulate1_beta_events.tsv"
beta_events2_path = beta_series_dir / "tables" / f"{subject}_modulate2_beta_events.tsv"

confounds_paths = {
    task: func_deriv_dir / f"{subject}_task-{task}_desc-confounds_timeseries.tsv"
    for task in tasks
}
confounds1_path = confounds_paths["modulate1"]
confounds2_path = confounds_paths["modulate2"]


# TODO: build run_id from timestamp + parameter label
run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_label = f"tR-{t_r:g}s__smooth-{smoothing_fwhm:g}mm__conf-{len(confound_columns)}"
run_id = f"{run_stamp}__{run_label}"

# TODO: create run_dir subdirectories
run_dir = beta_series_dir / "runs" / run_id
beta_maps_dir = run_dir / "beta_maps"
design_mats_dir = run_dir / "design_matrices"
figures_dir = run_dir / "figures"
provenance_dir = run_dir / "provenance"

for folder in [beta_maps_dir, design_mats_dir, figures_dir, provenance_dir]:
    folder.mkdir(parents=True, exist_ok=True)


for name, path in {
    "bold1": bold1_path,
    "bold2": bold2_path,
    "mask": mask_path,
    "beta_events1": beta_events1_path,
    "beta_events2": beta_events2_path,
    "confounds1": confounds1_path,
    "confounds2": confounds2_path,
}.items():
    print(name, path.exists(), path)


bold1 True C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
bold2 True C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.nii.gz
mask True C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz
beta_events1 True C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\tables\sub-097_modulate1_beta_events.tsv
beta_events2 True C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\tables\sub-097_modulate2_beta_events.tsv
confounds1 True C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_desc-confounds_timeseries.tsv
confounds2 True C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_desc-confounds_timeseries.tsv


---
## Section 2 — Save Provenance

Before running anything, write the parameters and input file paths to the provenance folder. Saving provenance first means that even if the model run crashes halfway through, you still have a record of what was attempted.

Writing the params dict to JSON:

```python
analysis_params = {
    "subject":        subject,
    "t_r":            t_r,
    "smoothing_fwhm": smoothing_fwhm,
    "hrf_model":      hrf_model,
    "confound_columns": confound_columns,
    "run_id":         run_id,
}

with open(provenance_dir / "params.json", "w") as f:
    json.dump(analysis_params, f, indent=2)
```

Saving the input file manifest as a TSV:

```python
inputs = pd.DataFrame([
    {"kind": "bold",      "task": "modulate1", "path": str(bold1_path)},
    {"kind": "bold",      "task": "modulate2", "path": str(bold2_path)},
    # ... add events, confounds, mask
])
inputs.to_csv(provenance_dir / "inputs.tsv", sep="\t", index=False)
```

**TODO:**
- [ ] Build `analysis_params` and write to `params.json`
- [ ] Build the inputs DataFrame and save to `inputs.tsv`
- [ ] Print `run_dir` so you can find the outputs later

In [12]:
# TODO: save params.json and inputs.tsv to provenance folder
analysis_params = {
    "subject": subject,
    "t_r": t_r,
    "smoothing_fwhm": smoothing_fwhm,
    "hrf_model": hrf_model,
    "confound_columns": confound_columns,
    "run_id": run_id
}

with open(provenance_dir / "params.json", "w") as f:
    json.dump(analysis_params, f, indent=2)

inputs = pd.DataFrame([
    {"kind": "bold", "task": "modulate1","path": str(bold1_path)},
    {"kind": "bold",      "task": "modulate2", "path": str(bold2_path)},
    {"kind": "events", "task": "modulate1", "path": str(beta_events1_path)},
    {"kind": "events", "task": "modulate2", "path": str(beta_events2_path)}
])

inputs.to_csv(provenance_dir / "inputs.tsv", sep= "\t", index = False)

---
## Section 3 — Load Events and Confounds

Load the labeled beta-series events (produced by notebook 01) and the confounds files. The files you are loading are:

**Beta-series events:**
```
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\sub-097_task-modulate1_events-betaSeries.tsv
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\sub-097_task-modulate2_events-betaSeries.tsv
```

**Confounds (full — you'll select columns from these):**
```
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate1_desc-confounds_timeseries.tsv
C:\ManzaRotation\Derivatives\sub-097\func\sub-097_task-modulate2_desc-confounds_timeseries.tsv
```

Loading and selecting confound columns:

```python
beta_events1 = pd.read_csv(beta_series_dir / f"{subject}_task-modulate1_events-betaSeries.tsv", sep="\t")

confounds1_full = pd.read_csv(func_deriv_dir / f"{subject}_task-modulate1_desc-confounds_timeseries.tsv", sep="\t")
confounds1 = confounds1_full[confound_columns]   # keep only the columns you want
```

Note: some confound columns (e.g., derivative columns) have a `NaN` in the first row because there's no prior timepoint to compute a derivative from. A common fix:

```python
confounds1 = confounds1.fillna(0)
```

**TODO:**
- [ ] Load `beta_events1`, `beta_events2`, `confounds1`, `confounds2`
- [ ] Select only `confound_columns` from each confounds DataFrame
- [ ] Handle NaNs with `.fillna(0)`
- [ ] Print shapes to confirm everything loaded correctly

In [22]:
# TODO: load beta_events1, beta_events2, confounds1, confounds2
beta_events1 = pd.read_csv(beta_events1_path, sep="\t")
beta_events2 = pd.read_csv(beta_events2_path, sep="\t")

confounds1_full = pd.read_csv(confounds1_path, sep="\t")
confounds2_full = pd.read_csv(confounds2_path, sep="\t")

confounds1 = confounds1_full[confound_columns].fillna(0)
confounds2 = confounds2_full[confound_columns].fillna(0)
print(confounds1_full.columns.tolist())

['global_signal', 'global_signal_derivative1', 'global_signal_derivative1_power2', 'global_signal_power2', 'csf', 'csf_derivative1', 'csf_derivative1_power2', 'csf_power2', 'white_matter', 'white_matter_derivative1', 'white_matter_power2', 'white_matter_derivative1_power2', 'csf_wm', 'tcompcor', 'std_dvars', 'dvars', 'framewise_displacement', 'rmsd', 't_comp_cor_00', 't_comp_cor_01', 't_comp_cor_02', 't_comp_cor_03', 'c_comp_cor_00', 'c_comp_cor_01', 'c_comp_cor_02', 'c_comp_cor_03', 'c_comp_cor_04', 'c_comp_cor_05', 'c_comp_cor_06', 'w_comp_cor_00', 'w_comp_cor_01', 'w_comp_cor_02', 'w_comp_cor_03', 'w_comp_cor_04', 'w_comp_cor_05', 'w_comp_cor_06', 'w_comp_cor_07', 'w_comp_cor_08', 'w_comp_cor_09', 'w_comp_cor_10', 'w_comp_cor_11', 'w_comp_cor_12', 'w_comp_cor_13', 'w_comp_cor_14', 'w_comp_cor_15', 'w_comp_cor_16', 'w_comp_cor_17', 'w_comp_cor_18', 'w_comp_cor_19', 'w_comp_cor_20', 'w_comp_cor_21', 'w_comp_cor_22', 'w_comp_cor_23', 'w_comp_cor_24', 'w_comp_cor_25', 'w_comp_cor_26', '

---
## Section 4 — Understand the LSS Events Builder (Conceptual Step)

Before writing the full loop, manually build the LSS design for **one trial** so the logic is clear in your head before you automate it.

**For a single trial of interest, you need to produce an events DataFrame where:**

| Row | `trial_type` | Description |
|---|---|---|
| Trial i | `trial_of_interest` | The specific feel trial you are currently estimating |
| Other trials in same condition | `other_{condition}` | Bundled nuisance regressor for the rest of that condition |
| Trials in other conditions | (original condition name, ungrouped) | Treated as separate condition-level regressors |

To extract the condition name from a trial label like `pos_val_01`, strip the numeric suffix. One way:

```python
trial_label = "pos_val_01"
condition   = "_".join(trial_label.split("_")[:-1])   # → "pos_val"
```

Then relabel each row based on whether it's the trial of interest, a same-condition trial, or a different-condition trial:

```python
def relabel(row_label, trial_label, condition):
    if row_label == trial_label:
        return "trial_of_interest"
    elif "_".join(row_label.split("_")[:-1]) == condition:
        return f"other_{condition}"
    else:
        return "_".join(row_label.split("_")[:-1])   # collapse to condition name
```

**TODO:**
- [ ] Pick the first trial from `beta_events1` as your trial of interest
- [ ] Apply the relabeling logic above manually (outside a function, just for one trial)
- [ ] Print the result and confirm it looks right before moving to Section 5

In [50]:
# Practice: build LSS events for a single trial
# Pick the first trial from beta_events1 as your trial of interest
trial_of_interest = beta_events1.loc[0, "trial_type"]
trial_label = trial_of_interest
condition = "_".join(trial_label.split("_")[:-1])

copy_of_beta1 = beta_events1.copy()

for idx in copy_of_beta1.index:
    row_label = copy_of_beta1.loc[idx, "trial_type"]
    row_condition = "_".join(row_label.split("_")[:-1])
    if copy_of_beta1.loc[idx, "trial_type"] == trial_label:
        copy_of_beta1.loc[idx, "trial_type"] = "trial_of_interest"
    elif row_condition == condition:
        copy_of_beta1.loc[idx, "trial_type"] = f"other_{condition}"
    else: 
        copy_of_beta1.loc[idx, "trial_type"] = row_condition


copy_of_beta1["trial_type"].value_counts()
       




trial_type
neg_val              8
pos_val              8
neg_aro              8
other_pos_aro        7
trial_of_interest    1
Name: count, dtype: int64

---
## Section 5 — Write the LSS Events Helper Function

Now wrap the logic from Section 4 into a reusable function. The function takes the full events DataFrame and one trial label, and returns the modified events DataFrame for that LSS GLM.

The skeleton is already in the code cell below. The key step inside is applying the relabeling to the `trial_type` column — pandas `.apply()` is a clean way to do this:

```python
lss_events = beta_events.copy()
lss_events["trial_type"] = lss_events["trial_type"].apply(
    lambda label: relabel(label, trial_label, condition)
)
```

**TODO:**
- [ ] Fill in `make_lss_events` using the relabeling logic from Section 4
- [ ] Test it on 2–3 different `trial_label` values
- [ ] Check: output always has same number of rows as input
- [ ] Check: exactly one row has `trial_type == "trial_of_interest"`

In [51]:
def make_lss_events(beta_events, trial_label):
    """
    Build the LSS events DataFrame for one trial.
    """

    # 1. Make a copy so the original beta_events table is not changed
    lss_events = beta_events.copy()

    # 2. Get the base condition of the target trial
    condition = "_".join(trial_label.split("_")[:-1])

    # 3. Loop through every row in the copied events DataFrame
    for idx in lss_events.index:
        # 4. Get this row's original trial label
        row_label = lss_events.loc[idx, "trial_type"]
        # 5. Get this row's base condition
        row_condition = "_".join(row_label.split("_")[:-1])

        # 6. If this row is the target trial,keep it as its original name
        if lss_events.loc[idx, "trial_type"] == trial_label:
            lss_events.loc[idx, "trial_type"] = trial_label

        # 7. Else if this row is the same condition as target,
        #    bundle it into the "other target condition" regressor
        elif row_condition == condition:
            lss_events.loc[idx, "trial_type"] = f"other_{condition}"
        # 8. Otherwise, collapse it back to its condition name
        else:
            lss_events.loc[idx, "trial_type"] = row_condition
            

    # 9. Return the modified LSS events table
    return lss_events

---
## Section 6 — Run the LSS Loop

Now run the full LSS loop. For each task run, iterate over every trial label and fit a separate GLM. Extract and save the beta map for `"trial_of_interest"`.

Organize the runs into a dict first so the loop is clean:

```python
runs = {
    "modulate1": {"bold": bold1_path, "events": beta_events1, "confounds": confounds1},
    "modulate2": {"bold": bold2_path, "events": beta_events2, "confounds": confounds2},
}
```

The nested loop structure — fitting one GLM per trial:

```python
for task, run_info in runs.items():
    trial_labels = run_info["events"]["trial_type"].tolist()

    for trial_label in trial_labels:
        print(f"  Fitting {task} | {trial_label}...")

        lss_events = make_lss_events(run_info["events"], trial_label)

        model = FirstLevelModel(t_r=t_r, mask_img=mask_img, smoothing_fwhm=smoothing_fwhm, hrf_model=hrf_model)
        model.fit(run_info["bold"], events=lss_events, confounds=run_info["confounds"])

        beta_map = model.compute_contrast("trial_of_interest", output_type="effect_size")

        out_name = f"{subject}_task-{task}_trial-{trial_label}_beta.nii.gz"
        beta_map.to_filename(beta_maps_dir / out_name)
```

Note: `output_type="effect_size"` gives you the raw beta estimate (parameter estimate), which is what you want for beta-series. `"z_score"` normalizes it — don't use that here.

**TODO:**
- [ ] Build the `runs` dict with both task runs
- [ ] Write the nested loop using the structure above
- [ ] Run it — expect 64 total GLM fits (32 trials × 2 runs), which will take a few minutes

In [ ]:
from nilearn.glm.first_level import FirstLevelModel

# TODO: set up the run configuration dict so you can loop over tasks cleanly
# runs = {
#     "modulate1": {"bold": bold1_path, "events": beta_events1, "confounds": confounds1},
#     "modulate2": {"bold": bold2_path, "events": beta_events2, "confounds": confounds2},
# }

# TODO: outer loop over task runs
#   TODO: inner loop over trial labels
#     TODO: make_lss_events, fit, compute_contrast, save


---
## Section 7 — Verify the Output Files

After the loop finishes, confirm you got the expected number of beta maps (32 per run × 2 runs = 64 total).

```python
beta_maps = sorted(beta_maps_dir.glob("*.nii.gz"))
print(f"Total beta maps: {len(beta_maps)}")   # expect 64

for p in beta_maps[:5]:                       # print a sample to check naming
    print(p.name)
```

**TODO:**
- [ ] Run the above and confirm you get 64 files
- [ ] Check that the filenames look correct (task and trial label are in the name)

In [ ]:
# TODO: glob for beta maps, count them, print a sample


---
## Section 8 — Quick Quality Check: Visualize a Few Beta Maps

Load a few beta maps and display them to confirm they look like plausible brain activation patterns.

Loading and plotting a saved beta map:

```python
import nibabel as nib

# Pick one map to inspect — grab any file from beta_maps_dir
example_path = beta_maps_dir / f"{subject}_task-modulate1_trial-pos_val_01_beta.nii.gz"
example_map  = nib.load(example_path)

plot_stat_map(example_map, threshold=0.5, title="pos_val_01 beta map")
show()
```

**What to look for:**
- Smooth, spatially coherent activation patterns (not speckled noise)
- Values within a reasonable range — not all zero, not extremely large
- No artifactual patterns like only the brain edges lighting up

**TODO:**
- [ ] Load and plot 2–4 beta maps from different conditions and/or runs
- [ ] Try `threshold=0.5` to see the spatial pattern more clearly

In [ ]:
from nilearn.plotting import plot_stat_map, show

# TODO: load and plot a few example beta maps


---
## Section 9 — Save Notebook to Provenance

Copy this notebook into the provenance folder so the exact code used for this run is permanently recorded alongside the outputs. `shutil.copy2` preserves the file metadata (timestamps) as well as the content.

```python
shutil.copy2(notebook_path, provenance_dir / notebook_path.name)
print(f"Saved to: {provenance_dir / notebook_path.name}")
```

**TODO:**
- [ ] Run the cell below — confirm the path printed exists inside the provenance folder

In [ ]:
notebook_path = Path(r"C:\ManzaRotation\Analysis\code\beta_series\03_beta_series_glm.ipynb")

# TODO: shutil.copy2(notebook_path, provenance_dir / notebook_path.name)
